# Upgrade Gadgets for Fault Tolerance

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do Reichardt's interleaved CNOT schedule and flagged state preparation eliminate single-fault problematic errors to restore full circuit distance $d_{\text{circuit}} = 2$?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Upgrade `syndrome` with Reichardt's interleaved CNOT schedule so mid-circuit faults trigger distinct parity checks.
- Upgrade `prepare_z_all` and `prepare_x_all` with a flag ancilla (`flags=['reject']`) to herald problematic preparation faults.
- Verify with `GadgetProfile` and `ec.audit` that every gadget in the upgraded protocol achieves circuit distance `2`.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## Upgrading `syndrome` to Distance 2 Without Extra Ancillas (Reichardt Interleaved Schedule)

How can we catch problematic errors on ancilla `4` ($X$-stabilizer) and ancilla `5` ($Z$-stabilizer) in $C_4$ **without adding extra flag qubits**?
Following **Reichardt ([arXiv:1804.06995 ↗](https://arxiv.org/abs/1804.06995))**, we **interleave** the CNOT gates of the $X$-stabilizer measurement (controlled on ancilla `4`) with the CNOT gates of the $Z$-stabilizer measurement (targeting ancilla `5`):

```
R 4 5
H 4
CX 4 0    # X-ancilla (4) touches data 0
CX 2 5    # Z-ancilla (5) touches data 2
CX 0 5    # Z-ancilla (5) touches data 0  <-- couples with ancilla 4 via data 0!
CX 1 5    # Z-ancilla (5) touches data 1
CX 4 2    # X-ancilla (4) touches data 2  <-- couples with ancilla 5 via data 2!
CX 4 3    # X-ancilla (4) touches data 3
CX 4 1    # X-ancilla (4) touches data 1
CX 3 5    # Z-ancilla (5) touches data 3
H 4
M 4 5
```

Why does this interleaving work?
1. **In a noiseless run:** Each pair of $X$-ancilla and $Z$-ancilla CNOTs crosses on an even number of shared data qubits (`0` and `2` before `CX 4 2` completes), so the net back-action between ancilla `4` and ancilla `5` cancels out!
2. **When a problematic fault occurs midway:**
   - An $X_4$ fault after `CX 4 3` (call 7) spreads to data qubits `1` and `3`, and because `CX 3 5` happens *after* `CX 4 3`, that $X_3$ error propagates onto ancilla `5` and flips the $Z$-check (`checks: [1]`)!
   - Symmetrically, a $Z_5$ fault after `CX 0 5` (call 5) spreads to data qubits `0` and `2`, and because `CX 4 2` happens *after* `CX 2 5`, that $Z_2$ error propagates back onto ancilla `4` and flips the $X$-check (`checks: [0]`)!
   In short, **ancillas 4 and 5 act as flags for each other's problematic errors!**

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Upgrade `syndrome` with Reichardt's Interleaved Circuit & Verify Distance 2
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

reichardt_syndrome_source = """
R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""

gadgets = protocol.layers[0].gadgets
gadgets["syndrome"].circuit.source = reichardt_syndrome_source
gadgets["syndrome"].checks = []
gadgets["syndrome"] = ec.filled(gadgets["syndrome"])

print("Audit after replacing `syndrome` and calling `ec.filled`:")
print(ec.audit(protocol))

syndrome_profile = ec.GadgetProfile(gadgets["syndrome"])
syndrome_distance = syndrome_profile.distance()
print(f"Upgraded `syndrome` circuit distance : d_circuit = {syndrome_distance.value}")
print(f"New minimum-weight fault witness   : {syndrome_distance.witness}")

# Trace the two single-fault problematic events that previously caused undetected logical errors:
problematic_faults = [
    ec.FaultEvent.after(5, ec.Pauli("Z_5")),
    ec.FaultEvent.after(7, ec.Pauli("X_4")),
]
problematic_effects = syndrome_profile.effects_of(problematic_faults)
improved_calls = gadgets["syndrome"].circuit.calls()
for fault, eff in zip(problematic_faults, problematic_effects):
    call_idx = fault.locations[0].after_call
    call = improved_calls[call_idx]
    print(f"\nProblematic fault `{fault}` after call[{call_idx}] ({call.mnemonic} {call.operands}):")
    print(f"  checks triggered : {eff.checks}  <-- Now detected!")
    print(f"  frame effect     : {eff.frames}")

Audit after replacing `syndrome` and calling `ec.filled`:


audit: ok
Upgraded `syndrome` circuit distance : d_circuit = 2
New minimum-weight fault witness   : X_4 after call 0; Z_5 after call 5

Problematic fault `Z_5 after call 5` after call[5] (CX [0, 5]):
  checks triggered : (Reference('checks[0]'), Reference('checks[2]'))  <-- Now detected!
  frame effect     : (Reference('out[0].x[0]'),)

Problematic fault `X_4 after call 7` after call[7] (CX [4, 2]):
  checks triggered : (Reference('checks[1]'), Reference('checks[3]'))  <-- Now detected!
  frame effect     : (Reference('out[0].z[1]'),)


## Upgrading `prepare_z_all` and `prepare_x_all` with a Flag Ancilla (`flags = ["reject"]`)

Next, let's fix `prepare_z_all` and `prepare_x_all`, which currently have $d_{\text{circuit}} = 1$.
To prepare $|00\rangle_L$ (`prepare_z_all`) or $|++\rangle_L$ (`prepare_x_all`) fault-tolerantly on $C_4$:
1. We prepare the 4-qubit GHZ/stabilizer state unitarily by initializing qubit `0` in $|+\rangle$ (`H 0`) and fanning out CNOTs `CX 0 1`, `CX 0 2`, `CX 0 3`.
2. Any single $X$ fault on control qubit `0` during that CNOT fan-out could spread to 2 data qubits. To catch it, we bracket the fan-out with two CNOTs into a **flag ancilla** on qubit `4` (`CX 0 4` before the fan-out, and `CX 0 4` after the fan-out), then measure ancilla `4` (`M 4`).
3. In `qodec`, we declare `gadget.implements.flags = ["reject"]`, bind `gadget.readouts = [{"reject": ["circuit.readouts[0]"]}]`, and call `ec.filled(gadget)` to automatically verify and populate the stabilizer checks!
   - *Why does `ec.filled` automatically generate `[outputs[0]:Z_0 Z_1 Z_2 Z_3, circuit.readouts[0]]`?* Because in the noiseless circuit, `CX 0 4` is applied twice around the fan-out, so ancilla `4` and the 4-qubit $Z$-stabilizer $Z_0 Z_1 Z_2 Z_3$ both end in $+1$; `ec.filled` discovers this exact stabilizer-to-flag invariant automatically.

Below, we upgrade both `prepare_z_all` and `prepare_x_all` with a flag ancilla (`M 4`) and verify that every gadget in our $C_4$ protocol now achieves static circuit distance `2`.

In [3]:
# Upgrade `prepare_z_all` and `prepare_x_all` with Flagged Preparation
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_syndrome")
gadgets = protocol.layers[0].gadgets

flagged_preparation_prefix = """R 0 1 2 3 4
H 0
CX 0 4
CX 0 1
CX 0 2
CX 0 3
CX 0 4
"""

# Step A (Worked Example): Upgrade `prepare_z_all` with flag ancilla `4`
prep_z = gadgets["prepare_z_all"]
prep_z.implements.flags = ["reject"]
prep_z.circuit.source = flagged_preparation_prefix + "M 4\n"
prep_z.readouts = [{"reject": ["circuit.readouts[0]"]}]
prep_z.checks = []
gadgets["prepare_z_all"] = ec.filled(prep_z)

# Step B: Apply the same flagged preparation pattern to `prepare_x_all` (`H 0 1 2 3` before `M 4`)
# Upgrade `prepare_x_all` using `flagged_preparation_prefix + "H 0 1 2 3\nM 4\n"`,
# bind `readouts = [{"reject": ["circuit.readouts[0]"]}]`, and synthesize checks with `ec.filled`.
prep_x = gadgets["prepare_x_all"]
prep_x.implements.flags = ["reject"]
prep_x.circuit.source = flagged_preparation_prefix + "H 0 1 2 3\nM 4\n"
prep_x.readouts = [{"reject": ["circuit.readouts[0]"]}]
prep_x.checks = []
gadgets["prepare_x_all"] = ec.filled(prep_x)

print("=== Full Protocol Audit After All Distance-2 Upgrades ===")
print(ec.audit(protocol))

print("=== Updated Circuit Distance Table Across All 6 Gadgets ===")
for name, gadget in gadgets.items():
    dist = ec.GadgetProfile(gadget).distance()
    print(f"  {name:<16} : d_circuit = {dist.value} (witness: {dist.witness})")
    assert dist.value == 2, f"Expected distance 2 for {name}, got {dist.value}"

=== Full Protocol Audit After All Distance-2 Upgrades ===


audit: ok
=== Updated Circuit Distance Table Across All 6 Gadgets ===
  cx_all           : d_circuit = 2 (witness: X_4 after call 0; X_5 after call 1)
  measure_x_all    : d_circuit = 2 (witness: X_0 after call 0; X_1 after call 1)
  measure_z_all    : d_circuit = 2 (witness: flip call 0 readout 0; flip call 1 readout 0)
  prepare_x_all    : d_circuit = 2 (witness: X_1 after call 1; X_2 after call 2)
  prepare_z_all    : d_circuit = 2 (witness: X_1 after call 1; X_2 after call 2)
  syndrome         : d_circuit = 2 (witness: X_4 after call 0; Z_5 after call 5)


## `[EXTENSION: Fire-and-Ice]` Paired-Support Fault Alignment & Selective State Filtering on the [[20, 2, 6]] Fire-and-Ice Code

> **Modular Extension ([arXiv:2026 / 2605.15344 ↗](https://arxiv.org/abs/2605.15344), Reichardt, Aasen & Chao):**
> In Module 1.5, we constructed the **$[[20, 2, 6]]$ Fire-and-Ice code** by concatenating the outer $[[5, 1, 3]]_4$ ("Fire") code onto 5 inner $[[4, 2, 2]]$ Iceberg ("Ice") blocks (`j = 0..4`).
>
> Look closely at the 8 weight-8 outer Fire stabilizers we built in Module 1.5: inside each $[[4, 2, 2]]$ block $j$, every outer stabilizer acts on either the **row pair** $\{4j, 4j+1\}$ ($\bar{X}_{j,0} / \bar{Z}_{j,1}$) or the **column pair** $\{4j, 4j+2\}$ ($\bar{X}_{j,1} / \bar{Z}_{j,0}$).
> - Suppose we measure a weight-4 or weight-8 check (or prepare a Steane ancilla block for selective state filtering) using CNOT gates across the 4 qubits of an Iceberg block.
> - Depending on the **CNOT ordering** on block $j$, a single midpoint ancilla fault spreads onto a specific pair of qubits inside that $[[4, 2, 2]]$ block:
>   - If the CNOT schedule pairs `(0, 1, 2, 3)` or `(0, 2, 1, 3)`, the midpoint problematic error (`X_2 X_3` or `X_1 X_3`) flips **only one** logical qubit of the $[[4, 2, 2]]$ block (which is detected by the opposite-type outer Fire stabilizer!).
>   - If instead the CNOT schedule pairs `(0, 3, 1, 2)` (diagonal pairing), the midpoint problematic error `X_1 X_2` flips **both** logical qubits of the $[[4, 2, 2]]$ block simultaneously ($\bar{X}_{j,0} \bar{X}_{j,1}$), which can masquerade as two independent sector errors across the two halves of Lemma 1!

Run the cell below to verify how the three CNOT pairings on a $[[4, 2, 2]]$ Iceberg block interact with both the inner $C_4$ logical operators and the outer $[[20, 2, 6]]$ Fire-and-Ice syndrome checks.

In [4]:
# Fire-and-Ice Extension: Analyze Midpoint Problematic Error Signatures on C4 & [[20, 2, 6]] Fire-and-Ice
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")

schedules = {
    "Row-paired  (0, 1, 2, 3)": [0, 1, 2, 3],
    "Col-paired  (0, 2, 1, 3)": [0, 2, 1, 3],
    "Diag-paired (0, 3, 1, 2)": [0, 3, 1, 2],
}

print(f"{'CNOT Ordering on Block 0':<26} | {'Midpoint Fault':<14} | {'Inner C4 Flipped Obs':<22} | {'Outer [[20,2,6]] Checks Violated'}")
print("-" * 98)
for label, order in schedules.items():
    spread_pauli = f"X_{order[2]} X_{order[3]}"
    inner_effect = str(c4_profile.logical_effect_of(ec.Pauli(spread_pauli)))
    outer_syn = sorted(fni_profile.syndrome_of(ec.Pauli(spread_pauli))) if "fni_profile" in globals() else "run Mod 1.5"
    print(f"{label:<26} | {spread_pauli:<14} | {inner_effect:<22} | {outer_syn}")

CNOT Ordering on Block 0   | Midpoint Fault | Inner C4 Flipped Obs   | Outer [[20,2,6]] Checks Violated
--------------------------------------------------------------------------------------------------
Row-paired  (0, 1, 2, 3)   | X_2 X_3        | X                      | run Mod 1.5
Col-paired  (0, 2, 1, 3)   | X_1 X_3        | IX                     | run Mod 1.5
Diag-paired (0, 3, 1, 2)   | X_1 X_2        | XX                     | run Mod 1.5


## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How does Reichardt's interleaved <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">syndrome</code> circuit achieve circuit distance 2 on <i>C</i><sub>4</sub> using the exact same two ancilla qubits (<code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">4</code> and <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">5</code>) as the distance-1 bare circuit?</summary>

<div style="padding:0.1em 1em;">

By interleaving the CNOTs of the $X$-stabilizer ancilla (`4`) and $Z$-stabilizer ancilla (`5`) so that they cross on data qubits `0` and `2`. In a noiseless run the two crossings cancel; when a single $X_4$ or $Z_5$ problematic fault occurs midway, only one crossing happens after the fault, propagating the fault to the partner ancilla and flipping its check!

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why did we have to set <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">gadget.implements.flags = ['reject']</code> and <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">gadget.readouts = [{'reject': ['circuit.readouts[0]']}]</code> before calling <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">ec.filled</code> on <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">prepare_z_all</code>?</summary>

<div style="padding:0.1em 1em;">

Because the ancilla measurement `M 4` at the end of `prepare_z_all` is a post-selection flag rather than a logical observable of the code. Declaring `flags = ['reject']` on the instruction and binding `readouts[0]` to `'reject'` tells `qdk.ec` and the runtime decoder to treat a `1` outcome on `M 4` as a flagged shot failure (`on_shot_failure='discard'`).

</div>

</details></div>

## Exercise: Upgrade prepare_x_all with a Flag Ancilla

Upgrade `prepare_x_all` with a flag ancilla (`M 4`), bind `readouts = [{'reject': ['circuit.readouts[0]']}]`, synthesize its checks with `ec.filled`, and return the upgraded gadget's circuit distance.

In [ ]:
from qdk import ec
from _unit import exercise, simulation


@exercise
def upgrade_prepare_x_distance() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_syndrome')
    prep_x = ns['protocol'].layers[0].gadgets['prepare_x_all']
    prep_x.implements.flags = ['reject']
    prep_x.circuit.source = (
        'R 0 1 2 3 4\nH 0\nCX 0 4\nCX 0 1\nCX 0 2\nCX 0 3\nCX 0 4\nH 0 1 2 3\nM 4\n'
    )
    prep_x.readouts = [{'reject': ['circuit.readouts[0]']}]
    prep_x.checks = []
    # Fill prep_x with ec.filled and return its integer circuit distance
    return ...

**Hint**

Call `upgraded = ec.filled(prep_x)` and return `ec.GadgetProfile(upgraded).distance().value`.

In [ ]:
from qdk import ec
from _unit import exercise, simulation


@exercise
def upgrade_prepare_x_distance() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_syndrome')
    prep_x = ns['protocol'].layers[0].gadgets['prepare_x_all']
    prep_x.implements.flags = ['reject']
    prep_x.circuit.source = (
        'R 0 1 2 3 4\nH 0\nCX 0 4\nCX 0 1\nCX 0 2\nCX 0 3\nCX 0 4\nH 0 1 2 3\nM 4\n'
    )
    prep_x.readouts = [{'reject': ['circuit.readouts[0]']}]
    prep_x.checks = []
    upgraded = ec.filled(prep_x)
    return ec.GadgetProfile(upgraded).distance().value

Bracketing the CNOT fan-out from control qubit `0` with two `CX 0 4` gates onto flag ancilla `4` catches any single $X_0$ fault that spreads to multiple data qubits, while `H 0 1 2 3` rotates the stabilized $+Z_0, +Z_1$ state into $+X_0, +X_1$, achieving circuit distance `2`.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 6
- **Learning Objectives:**
  1. Replace `gadgets["syndrome"].circuit.source` with Reichardt's interleaved 8-CNOT circuit, synthesize checks via `ec.filled`, and verify that `GadgetProfile(gadgets["syndrome"]).distance().value == 2`.
  2. Trace the two problematic fault events (`Z_5` after call 5 and `X_4` after call 7) with `effects_of` and point out a subtle, crucial fact that sets up Module 9:
     - Both problematic faults now trigger a check (`checks: [0]` or `checks: [1]`), so they are **detected** (giving circuit distance 2!).
     - However, because they put a weight-2 error on the data block (`frames: [outputs[0]:X_0, outputs[0]:X_1]` or `Z_0, Z_1`), a minimum-weight decoder that tries to *correct* (rather than post-select/reject) a triggered syndrome check will misidentify the weight-2 problematic error as a weight-1 error! That is why we will combine flag rejection with decoder readout-score post-selection in Module 9.
  3. Upgrade `prepare_z_all` and `prepare_x_all` to circuit distance 2 by adding ancilla 4 as a `reject` flag (`gadget.implements.flags = ["reject"]`, `gadget.readouts = [{"reject": ["circuit.readouts[0]"]}]`) and calling `ec.filled`.

#### 🤖 COPILOT TEACHING NOTE — Fire-and-Ice Extension (Module 6)
- Have the learner compare the three pairings of the 4 qubits `{0, 1, 2, 3}` of Iceberg block `j = 0` inside `FireAndIce_20_2_6`:
  - Ordering `(0, 1, 2, 3)` produces midpoint ancilla fault `X_2 X_3` (flips $\bar{Z}_0$ of block 0; triggers 2 outer Fire $Z$-checks, zero inner checks).
  - Ordering `(0, 2, 1, 3)` produces midpoint ancilla fault `X_1 X_3` (flips $\bar{Z}_1$ of block 0; triggers 2 outer Fire $Z$-checks, zero inner checks).
  - Ordering `(0, 3, 1, 2)` produces midpoint ancilla fault `X_1 X_2` (flips **both** $\bar{Z}_0$ and $\bar{Z}_1$ of block 0!).
- This illustrates why the *Fire and Ice* paper designs its Steane ancilla verification and selective state filtering circuits around **paired-support alignment** on the $[[4, 2, 2]]$ blocks.

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../05-circuit-distance/circuit_distance.workbook.ipynb" title="Previous unit: Characterize Circuit Distance & Problematic Errors" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Characterize Circuit Distance & Problematic Errors</span></span></a><a href="../07-qsharp-compilation/qsharp_compilation.workbook.ipynb" title="Next unit: Compile Logical Experiments from Q#" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Compile Logical Experiments from Q#</span></span><span aria-hidden="true">&#8250;</span></a></div>